# Full-Study Detectability Classifier (Finding v)

Trains classifiers to predict each hallucination type (H1-H6 + ANY) from the model output,
using two parallel embedding methods for robustness:

1. **sentence-transformers `all-MiniLM-L6-v2`** — fast, free, CPU-runnable
2. **OpenAI `text-embedding-3-small`** — higher dimension (1536), generally stronger

For each embedding type, we report:
- 5-fold CV AUC and F1 per H type (LogReg, RF, GB)
- Held-out 80/20 evaluation per H type
- Hand-crafted feature set ablations (verbosity, hedging, refusal flag, etc.)

If both embedding methods give similar AUC patterns per H type, Finding v is robust.

**Inputs:**
- `all_triplets_cache.csv` — model outputs
- `panel_raw_judge_labels_full.csv` — H1-H6 labels

**Outputs (saved to `C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\`):**
- `embeddings_minilm.npy`, `embeddings_openai.npy` — caches (re-runs reuse)
- `embedding_results_minilm.json`, `embedding_results_openai.json` — 5-fold CV
- `held_out_results.json` — test-set performance, both methods side-by-side
- `feature_importance.json` — most predictive hand-crafted features per H type
- `detectability_summary.txt` — paper-ready


In [1]:
#!pip install sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer; print('ok')

ok


In [3]:
import os, json, re, time
import numpy as np
import pandas as pd

OUTPUT_DIR    = r'C:\Opeyemi\PROMPTS\EVALUATION'
TRIPLETS_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')
PANEL_RAW     = os.path.join(OUTPUT_DIR, 'panel_raw_judge_labels_full.csv')

API_KEYS_DIR = r'C:\Opeyemi\PROMPTS\API-KEYS'
OPENAI_KEY_PATH = os.path.join(API_KEYS_DIR, 'chatgpt.txt')

HALL_DIR = r'C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS'
os.makedirs(HALL_DIR, exist_ok=True)

EMB_MINILM_PATH   = os.path.join(HALL_DIR, 'embeddings_minilm.npy')
EMB_OPENAI_PATH   = os.path.join(HALL_DIR, 'embeddings_openai.npy')
TRIPLET_INDEX     = os.path.join(HALL_DIR, 'embedding_index.csv')
EMB_RESULTS_MINILM = os.path.join(HALL_DIR, 'embedding_results_minilm.json')
EMB_RESULTS_OPENAI = os.path.join(HALL_DIR, 'embedding_results_openai.json')
HELDOUT_RESULTS    = os.path.join(HALL_DIR, 'held_out_results.json')
FEATURE_IMPORT     = os.path.join(HALL_DIR, 'feature_importance.json')
DETECT_SUMMARY     = os.path.join(HALL_DIR, 'detectability_summary.txt')

HCOLS = ['H1','H2','H3','H4','H5','H6']
RANDOM_SEED = 42

# Load triplets
print('Loading triplets...')
df_trip = pd.read_csv(TRIPLETS_PATH)
df_trip['model_output'] = df_trip['model_output'].fillna('').astype(str)
print(f'  Triplets: {len(df_trip):,}')

# Load panel labels and aggregate to majority verdict per triplet
print('Loading panel labels...')
labels = pd.read_csv(PANEL_RAW)
for h in HCOLS:
    labels[h] = pd.to_numeric(labels[h], errors='coerce')
clean = labels[(labels[HCOLS] >= 0).all(axis=1)].copy()

def majority(col):
    return col.mode().iloc[0] if not col.mode().empty else 0

mv = clean.groupby(['row_idx','model','technique','video','crime_type'])[HCOLS].agg(majority).reset_index()
print(f'  Panel verdicts (majority): {len(mv):,}')

# Add ANY column
mv['ANY'] = (mv[HCOLS].sum(axis=1) > 0).astype(int)

# Merge labels onto triplets — note: we want one row per (model, technique, video) with both
# the model_output and the H1..H6+ANY labels. Use the index as a stable key.
df_trip = df_trip.reset_index().rename(columns={'index': 'row_idx'})
data = df_trip.merge(mv[['row_idx'] + HCOLS + ['ANY']], on='row_idx', how='inner')
print(f'  Merged dataset (output + labels): {len(data):,}')

# Save the index so embeddings and labels stay aligned
data[['row_idx','model','technique','video','crime_type']].to_csv(TRIPLET_INDEX, index=False)
print(f'  Saved index: {TRIPLET_INDEX}')

# Truncate model outputs to a reasonable length for embedding (most embedders accept ~8K tokens
# but this keeps API cost predictable). 4000 chars is roughly 1000 tokens, plenty for our task.
data['embed_text'] = data['model_output'].str.slice(0, 4000)

print(f'\nFinal dataset: {len(data):,} rows × {len(HCOLS)+1} target columns')
print('Per-H positive rate:')
for h in HCOLS + ['ANY']:
    print(f'  {h}: {data[h].mean()*100:.1f}% ({int(data[h].sum())} of {len(data)})')


Loading triplets...
  Triplets: 9,680
Loading panel labels...
  Panel verdicts (majority): 9,680
  Merged dataset (output + labels): 9,680
  Saved index: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\embedding_index.csv

Final dataset: 9,680 rows × 7 target columns
Per-H positive rate:
  H1: 54.6% (5282 of 9680)
  H2: 18.5% (1787 of 9680)
  H3: 39.8% (3848 of 9680)
  H4: 25.1% (2426 of 9680)
  H5: 60.3% (5839 of 9680)
  H6: 23.7% (2294 of 9680)
  ANY: 85.8% (8305 of 9680)


---
## 1. Compute MiniLM embeddings (cached)

In [4]:
# sentence-transformers (CPU is fine for MiniLM, ~30 min for 9,680 outputs)
# If not installed: pip install sentence-transformers
import numpy as np

if os.path.exists(EMB_MINILM_PATH):
    emb_minilm = np.load(EMB_MINILM_PATH)
    print(f'Loaded cached MiniLM embeddings: {emb_minilm.shape}')
    if len(emb_minilm) != len(data):
        print(f'WARNING: cache size {len(emb_minilm)} != dataset size {len(data)}.')
        print('Delete the .npy and re-run if you regenerated the dataset.')
else:
    print('Computing MiniLM embeddings (this may take ~20-30 min on CPU)...')
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        raise SystemExit('Run: pip install sentence-transformers')

    model = SentenceTransformer('all-MiniLM-L6-v2')
    texts = data['embed_text'].tolist()
    emb_minilm = model.encode(
        texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(EMB_MINILM_PATH, emb_minilm)
    print(f'Saved: {EMB_MINILM_PATH}  shape={emb_minilm.shape}')


Computing MiniLM embeddings (this may take ~20-30 min on CPU)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Julia\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Julia\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/152 [00:00<?, ?it/s]

Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\embeddings_minilm.npy  shape=(9680, 384)


---
## 2. Compute OpenAI embeddings (cached)

In [5]:
# OpenAI text-embedding-3-small: 1536-dim, ~$0.02 per 1M tokens.
# 9,680 outputs × ~1000 tokens ≈ 9.7M tokens ≈ $0.20.

if os.path.exists(EMB_OPENAI_PATH):
    emb_openai = np.load(EMB_OPENAI_PATH)
    print(f'Loaded cached OpenAI embeddings: {emb_openai.shape}')
else:
    print('Computing OpenAI embeddings...')
    import openai

    with open(OPENAI_KEY_PATH) as f:
        key = f.read().strip()
    client = openai.OpenAI(api_key=key)

    texts = data['embed_text'].tolist()
    embeddings = []
    BATCH = 100  # OpenAI accepts up to 2048 inputs per request, but 100 is a safe size

    from tqdm import tqdm
    for start in tqdm(range(0, len(texts), BATCH), desc='OpenAI batches'):
        batch = texts[start:start+BATCH]
        # Replace empty strings (API rejects them)
        batch = [t if t.strip() else '[empty]' for t in batch]
        for attempt in range(5):
            try:
                resp = client.embeddings.create(model='text-embedding-3-small', input=batch)
                for item in resp.data:
                    embeddings.append(item.embedding)
                break
            except Exception as e:
                err = str(e)
                if '429' in err or 'rate' in err.lower():
                    wait = 30 * (attempt + 1)
                    print(f'  rate-limited, sleeping {wait}s')
                    time.sleep(wait)
                else:
                    print(f'  ERROR on attempt {attempt+1}: {err[:200]}')
                    time.sleep(5)
                    if attempt == 4:
                        raise

    emb_openai = np.array(embeddings, dtype=np.float32)
    # Normalize
    norms = np.linalg.norm(emb_openai, axis=1, keepdims=True)
    emb_openai = emb_openai / np.clip(norms, 1e-9, None)
    np.save(EMB_OPENAI_PATH, emb_openai)
    print(f'\nSaved: {EMB_OPENAI_PATH}  shape={emb_openai.shape}')

print(f'\nMiniLM dim:  {emb_minilm.shape[1]}')
print(f'OpenAI dim:  {emb_openai.shape[1]}')


Computing OpenAI embeddings...


OpenAI batches: 100%|██████████| 97/97 [01:12<00:00,  1.33it/s]



Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\embeddings_openai.npy  shape=(9680, 1536)

MiniLM dim:  384
OpenAI dim:  1536


---
## 3. Hand-crafted features

Pilot's `retrained_classifier_results.json` and `ablation_results.json` used a small set
of structural features on top of (or instead of) embeddings:
- **verbosity** (word count)
- **hedging** word density (e.g. "may", "possibly", "appears", "suggests")
- **refusal indicators**
- **claim density**
- **model one-hot** (Claude / GPT / Gemini)
- **technique one-hot** (Zero-Shot / Sequential / Least-to-Most / ReAct)
- **multi-turn flag**

In [6]:
HEDGE_WORDS = [
    ' may ', ' might ', ' possibly ', ' perhaps ', ' likely ', ' probably ',
    ' appears ', ' seems ', ' suggests ', ' suggesting ', ' could ',
    ' presumably ', ' apparently ', ' uncertain', ' approximately ',
    ' roughly ', ' somewhat ', ' arguably ',
]
REFUSAL_PHRASES = [
    'i cannot', "i can't", "i'm unable", 'i am unable', "i'm not able",
    'cannot analyze', 'cannot process', 'not appropriate', 'i must decline',
    'unable to provide', 'cannot provide', 'i refuse',
    "i'm sorry, but", 'against my guidelines',
]
SENT_RE  = re.compile(r'[.!?]+(?:\s|$)')
CLAIM_RE = re.compile(r'(?:[.!?;]|--|—|\n\s*[-*•])')

def hand_features(text, model_name, technique):
    if not isinstance(text, str):
        text = ''
    t = text.lower()
    words = text.split()
    nw = len(words)
    n_sent = sum(1 for p in SENT_RE.split(text) if p.strip())
    n_claim = sum(1 for p in CLAIM_RE.split(text) if len(p.split()) >= 3)

    n_hedge = sum(t.count(h) for h in HEDGE_WORDS)
    is_refusal = int(any(p in t for p in REFUSAL_PHRASES))

    out = {
        'verbosity_words':      nw,
        'verbosity_log':        np.log1p(nw),
        'hedging_density':      n_hedge / max(nw, 1),
        'refusal_flag':         is_refusal,
        'claim_density':        n_claim / max(nw, 1),
        'sent_density':         n_sent / max(nw, 1),
        # Model one-hot
        'is_claude':            int(model_name == 'Claude'),
        'is_gpt':               int(model_name == 'GPT'),
        'is_gemini':            int(model_name == 'Gemini'),
        # Technique one-hot
        'is_zero_shot':         int(technique == 'Zero-Shot'),
        'is_sequential':        int(technique == 'Sequential'),
        'is_least_to_most':     int(technique == 'Least-to-Most'),
        'is_react':             int(technique == 'ReAct'),
        # Multi-turn flag
        'is_multiturn':         int(technique in ('Sequential','Least-to-Most','ReAct')),
    }
    return out

print('Computing hand-crafted features...')
feats_list = [hand_features(row['model_output'], row['model'], row['technique'])
              for _, row in data.iterrows()]
hand_df = pd.DataFrame(feats_list)
print(f'Hand features shape: {hand_df.shape}')
print(f'Feature names: {list(hand_df.columns)}')

X_hand = hand_df.values.astype(np.float32)


Computing hand-crafted features...
Hand features shape: (9680, 14)
Feature names: ['verbosity_words', 'verbosity_log', 'hedging_density', 'refusal_flag', 'claim_density', 'sent_density', 'is_claude', 'is_gpt', 'is_gemini', 'is_zero_shot', 'is_sequential', 'is_least_to_most', 'is_react', 'is_multiturn']


---
## 4. Train classifiers (5-fold CV) for both embedding sets

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler

CLASSIFIERS = {
    'LogReg': lambda: LogisticRegression(max_iter=2000, class_weight='balanced',
                                         random_state=RANDOM_SEED, C=1.0),
    'RF':     lambda: RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                             random_state=RANDOM_SEED,
                                             class_weight='balanced'),
    'GB':     lambda: GradientBoostingClassifier(n_estimators=150, max_depth=3,
                                                  random_state=RANDOM_SEED),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def train_cv(X, y, label):
    """5-fold CV: returns mean F1 and AUC across folds, picks best CLF."""
    if y.sum() < 10 or y.sum() > len(y) - 10:
        return {'note': f'class imbalance too severe (n_pos={int(y.sum())})'}
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    best = None
    per_clf = {}
    for name, ctor in CLASSIFIERS.items():
        try:
            scores = cross_validate(
                ctor(), X_scaled, y, cv=skf, n_jobs=-1,
                scoring=('f1','roc_auc'), error_score='raise',
            )
            per_clf[name] = {
                'f1':  float(np.mean(scores['test_f1'])),
                'auc': float(np.mean(scores['test_roc_auc'])),
            }
        except Exception as e:
            per_clf[name] = {'error': str(e)[:200]}
    # Pick best by AUC
    valid = {k: v for k, v in per_clf.items() if 'auc' in v}
    if valid:
        best_clf = max(valid, key=lambda k: valid[k]['auc'])
        best = {'clf': best_clf, **valid[best_clf]}
    return {'per_clf': per_clf, 'best': best}

def run_full_eval(X, name, save_path):
    print(f'\n=== {name} embeddings: shape {X.shape} ===')
    results = {}
    for h in HCOLS + ['ANY']:
        y = data[h].astype(int).values
        n_pos = int(y.sum())
        print(f'  {h}: n_pos={n_pos:,}', end=' ', flush=True)
        r = train_cv(X, y, h)
        results[h] = r
        if 'best' in r and r['best']:
            print(f"-> best={r['best']['clf']} AUC={r['best']['auc']:.3f} F1={r['best']['f1']:.3f}")
        else:
            print(f"-> {r}")
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved: {save_path}')
    return results

results_minilm = run_full_eval(emb_minilm, 'MiniLM', EMB_RESULTS_MINILM)
results_openai = run_full_eval(emb_openai, 'OpenAI', EMB_RESULTS_OPENAI)

# Print side-by-side AUC comparison
print('\n\n=== AUC comparison: MiniLM vs OpenAI (5-fold CV best classifier) ===')
print(f'{"H type":<6}{"MiniLM AUC":>14}{"OpenAI AUC":>14}{"Δ":>10}')
print('-' * 44)
for h in HCOLS + ['ANY']:
    m = results_minilm.get(h, {}).get('best', {})
    o = results_openai.get(h, {}).get('best', {})
    if m and o:
        m_auc, o_auc = m['auc'], o['auc']
        print(f'{h:<6}{m_auc:>14.3f}{o_auc:>14.3f}{(o_auc-m_auc):>+10.3f}')



=== MiniLM embeddings: shape (9680, 384) ===
  H1: n_pos=5,282 -> best=RF AUC=0.841 F1=0.776
  H2: n_pos=1,787 -> best=RF AUC=0.765 F1=0.046
  H3: n_pos=3,848 -> best=RF AUC=0.815 F1=0.623
  H4: n_pos=2,426 -> best=RF AUC=0.838 F1=0.424
  H5: n_pos=5,839 -> best=RF AUC=0.845 F1=0.815
  H6: n_pos=2,294 -> best=RF AUC=0.821 F1=0.297
  ANY: n_pos=8,305 -> best=RF AUC=0.770 F1=0.924
Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\embedding_results_minilm.json

=== OpenAI embeddings: shape (9680, 1536) ===
  H1: n_pos=5,282 -> best=RF AUC=0.854 F1=0.779
  H2: n_pos=1,787 -> best=RF AUC=0.808 F1=0.172
  H3: n_pos=3,848 -> best=GB AUC=0.860 F1=0.729
  H4: n_pos=2,426 -> best=RF AUC=0.867 F1=0.555
  H5: n_pos=5,839 -> best=RF AUC=0.857 F1=0.821
  H6: n_pos=2,294 -> best=RF AUC=0.844 F1=0.458
  ANY: n_pos=8,305 -> best=RF AUC=0.808 F1=0.923
Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\embedding_results_openai.json


=== AUC comparison: MiniLM vs OpenAI (5-fold CV best classifier) ===
H type

---
## 5. Held-out evaluation (80/20 stratified split)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

heldout = {}
for emb_name, X in [('minilm', emb_minilm), ('openai', emb_openai)]:
    print(f'\n=== Held-out {emb_name} ===')
    heldout[emb_name] = {}
    for h in HCOLS + ['ANY']:
        y = data[h].astype(int).values
        n_pos = int(y.sum())
        if n_pos < 20 or n_pos > len(y) - 20:
            heldout[emb_name][h] = {'note': f'imbalanced (n_pos={n_pos})'}
            continue
        # Stratified split
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y,
        )
        scaler = StandardScaler().fit(Xtr)
        Xtr_s = scaler.transform(Xtr)
        Xte_s = scaler.transform(Xte)

        # Use the best CV classifier for this H (or default to LogReg)
        cv_res = (results_minilm if emb_name == 'minilm' else results_openai)
        best_name = cv_res.get(h, {}).get('best', {}).get('clf', 'LogReg')
        clf = CLASSIFIERS[best_name]()
        clf.fit(Xtr_s, ytr)
        ypred = clf.predict(Xte_s)
        try:
            yprob = clf.predict_proba(Xte_s)[:, 1]
            auc = float(roc_auc_score(yte, yprob))
        except Exception:
            auc = None
        prec, rec, f1, _ = precision_recall_fscore_support(yte, ypred, average='binary',
                                                            zero_division=0)
        heldout[emb_name][h] = {
            'clf': best_name,
            'prec': float(prec), 'rec': float(rec), 'f1': float(f1),
            'auc': auc,
            'n_pos_test': int(yte.sum()),
            'n_test': int(len(yte)),
        }
        print(f'  {h}: clf={best_name}  AUC={auc:.3f}  F1={f1:.3f}  P={prec:.3f}  R={rec:.3f}  '
              f'(n_pos_test={int(yte.sum())})')

with open(HELDOUT_RESULTS, 'w') as f:
    json.dump(heldout, f, indent=2)
print(f'\nSaved: {HELDOUT_RESULTS}')



=== Held-out minilm ===
  H1: clf=RF  AUC=0.837  F1=0.763  P=0.753  R=0.773  (n_pos_test=1056)
  H2: clf=RF  AUC=0.769  F1=0.048  P=0.529  R=0.025  (n_pos_test=357)
  H3: clf=RF  AUC=0.815  F1=0.637  P=0.806  R=0.527  (n_pos_test=770)
  H4: clf=RF  AUC=0.833  F1=0.430  P=0.753  R=0.301  (n_pos_test=485)
  H5: clf=RF  AUC=0.821  F1=0.794  P=0.727  R=0.875  (n_pos_test=1168)
  H6: clf=RF  AUC=0.825  F1=0.283  P=0.711  R=0.176  (n_pos_test=459)
  ANY: clf=RF  AUC=0.769  F1=0.923  P=0.858  R=0.998  (n_pos_test=1661)

=== Held-out openai ===
  H1: clf=RF  AUC=0.845  F1=0.766  P=0.797  R=0.738  (n_pos_test=1056)
  H2: clf=RF  AUC=0.831  F1=0.130  P=0.619  R=0.073  (n_pos_test=357)
  H3: clf=GB  AUC=0.865  F1=0.739  P=0.792  R=0.692  (n_pos_test=770)
  H4: clf=RF  AUC=0.861  F1=0.547  P=0.782  R=0.421  (n_pos_test=485)
  H5: clf=RF  AUC=0.834  F1=0.798  P=0.751  R=0.851  (n_pos_test=1168)
  H6: clf=RF  AUC=0.847  F1=0.472  P=0.740  R=0.346  (n_pos_test=459)
  ANY: clf=RF  AUC=0.797  F1=0.925

---
## 6. Hand-crafted feature classifier + ablations

Tests how much of the AUC comes from embeddings vs structural features.

In [ ]:
# Hand-crafted features only (no embeddings)
print('=== Hand-crafted features only ===')
hand_results = {}
for h in HCOLS + ['ANY']:
    y = data[h].astype(int).values
    r = train_cv(X_hand, y, h)
    hand_results[h] = r
    if r.get('best'):
        print(f"  {h}: best={r['best']['clf']} AUC={r['best']['auc']:.3f} "
              f"F1={r['best']['f1']:.3f}")

# Combined: embeddings + hand features
print('\n=== Combined: MiniLM + hand features ===')
X_combined_minilm = np.hstack([emb_minilm, X_hand])
combined_minilm_results = {}
for h in HCOLS + ['ANY']:
    y = data[h].astype(int).values
    r = train_cv(X_combined_minilm, y, h)
    combined_minilm_results[h] = r
    if r.get('best'):
        print(f"  {h}: AUC={r['best']['auc']:.3f}")

print('\n=== Combined: OpenAI + hand features ===')
X_combined_openai = np.hstack([emb_openai, X_hand])
combined_openai_results = {}
for h in HCOLS + ['ANY']:
    y = data[h].astype(int).values
    r = train_cv(X_combined_openai, y, h)
    combined_openai_results[h] = r
    if r.get('best'):
        print(f"  {h}: AUC={r['best']['auc']:.3f}")

# Feature ablation: hand features in/out of selected groups
print('\n=== Hand-feature ablation (LogReg, AUC per H) ===')
FEATURE_GROUPS = {
    'all':              list(hand_df.columns),
    'minus_verbosity':  [c for c in hand_df.columns if 'verbosity' not in c],
    'minus_hedging':    [c for c in hand_df.columns if 'hedging' not in c],
    'minus_refusal':    [c for c in hand_df.columns if 'refusal' not in c],
    'minus_model':      [c for c in hand_df.columns if not c.startswith('is_claude')
                         and not c.startswith('is_gpt') and not c.startswith('is_gemini')],
    'minus_technique':  [c for c in hand_df.columns if not c.startswith('is_zero')
                         and not c.startswith('is_seq') and not c.startswith('is_least')
                         and not c.startswith('is_react') and not c.startswith('is_multi')],
    'only_verbosity':   [c for c in hand_df.columns if 'verbosity' in c],
    'only_hedging':     [c for c in hand_df.columns if 'hedging' in c],
    'only_refusal':     [c for c in hand_df.columns if 'refusal' in c],
    'only_model':       [c for c in hand_df.columns if c.startswith('is_claude')
                         or c.startswith('is_gpt') or c.startswith('is_gemini')],
    'only_multiturn':   [c for c in hand_df.columns if c == 'is_multiturn'],
}

ablation = {}
for group_name, cols in FEATURE_GROUPS.items():
    if not cols:
        continue
    X_sub = hand_df[cols].values.astype(np.float32)
    per_h = {}
    for h in HCOLS + ['ANY']:
        y = data[h].astype(int).values
        if y.sum() < 10:
            continue
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X_sub)
        try:
            scores = cross_validate(
                LogisticRegression(max_iter=2000, class_weight='balanced',
                                   random_state=RANDOM_SEED),
                X_s, y, cv=skf, scoring='roc_auc', n_jobs=-1,
            )
            per_h[h] = float(np.mean(scores['test_roc_auc']))
        except Exception:
            per_h[h] = None
    ablation[group_name] = per_h

# Save
feature_results = {
    'hand_only_5fold_cv': hand_results,
    'combined_minilm':    combined_minilm_results,
    'combined_openai':    combined_openai_results,
    'hand_feature_ablation_LogReg_auc': ablation,
}
with open(FEATURE_IMPORT, 'w') as f:
    json.dump(feature_results, f, indent=2)
print(f'\nSaved: {FEATURE_IMPORT}')

# Pretty-print ablation table
print('\nHand-feature ablation table (LogReg AUC):')
print(f'{"Feature group":<22}', end='')
for h in HCOLS + ['ANY']:
    print(f'{h:>7}', end='')
print()
print('-' * (22 + 7*7))
for group, per_h in ablation.items():
    print(f'{group:<22}', end='')
    for h in HCOLS + ['ANY']:
        v = per_h.get(h)
        print(f'{v:>7.3f}' if v is not None else f'{"--":>7}', end='')
    print()


=== Hand-crafted features only ===
  H1: best=GB AUC=0.833 F1=0.740
  H2: best=GB AUC=0.709 F1=0.019
  H3: best=GB AUC=0.749 F1=0.583
  H4: best=GB AUC=0.782 F1=0.437
  H5: best=GB AUC=0.846 F1=0.809
  H6: best=GB AUC=0.781 F1=0.360
  ANY: best=GB AUC=0.762 F1=0.922

=== Combined: MiniLM + hand features ===
  H1: AUC=0.849
  H2: AUC=0.770
  H3: AUC=0.819
  H4: AUC=0.840
  H5: AUC=0.854
  H6: AUC=0.827
  ANY: AUC=0.780

=== Combined: OpenAI + hand features ===
  H1: AUC=0.854
  H2: AUC=0.811
  H3: AUC=0.860
  H4: AUC=0.866


---
## 7. Paper-ready summary

In [ ]:
lines = []
lines.append('=' * 76)
lines.append('FULL-STUDY DETECTABILITY SUMMARY (Finding v)')
lines.append('=' * 76)
lines.append('')
lines.append(f'Dataset: {len(data):,} model outputs')
lines.append(f'Per-H positive rates:')
for h in HCOLS + ['ANY']:
    pr = data[h].mean() * 100
    lines.append(f'  {h}: {pr:>5.1f}%  (n_pos={int(data[h].sum()):,})')
lines.append('')
lines.append('-- 5-FOLD CV AUC (best classifier per H) --')
lines.append(f'{"H":<6}{"MiniLM":>10}{"OpenAI":>10}{"HandOnly":>10}{"Comb_OAI":>10}')
lines.append('-' * 46)
for h in HCOLS + ['ANY']:
    m = results_minilm.get(h, {}).get('best', {})
    o = results_openai.get(h, {}).get('best', {})
    hh = hand_results.get(h, {}).get('best', {})
    co = combined_openai_results.get(h, {}).get('best', {})
    line = f'{h:<6}'
    for r in (m, o, hh, co):
        line += f'{r["auc"]:>10.3f}' if r and 'auc' in r else f'{"--":>10}'
    lines.append(line)
lines.append('')
lines.append('-- HELD-OUT 80/20 (best per H) --')
lines.append(f'{"H":<6}{"MiniLM AUC":>14}{"MiniLM F1":>14}{"OpenAI AUC":>14}{"OpenAI F1":>14}')
lines.append('-' * 62)
for h in HCOLS + ['ANY']:
    m = heldout.get('minilm', {}).get(h, {})
    o = heldout.get('openai', {}).get(h, {})
    line = f'{h:<6}'
    for r in (m, o):
        if r and 'auc' in r and r['auc'] is not None:
            line += f'{r["auc"]:>14.3f}{r["f1"]:>14.3f}'
        else:
            line += f'{"--":>14}{"--":>14}'
    lines.append(line)

lines.append('')
lines.append('-- HEADLINE NUMBER FOR ABSTRACT --')
# Find the maximum AUC across all H types from the best embedding+ablation
best_auc_h, best_auc_v = None, 0
for h in HCOLS:
    for src in (results_openai, results_minilm, combined_openai_results):
        v = src.get(h, {}).get('best', {}).get('auc', 0)
        if v and v > best_auc_v:
            best_auc_v, best_auc_h = v, h
lines.append(f'  Best detectability: {best_auc_h} AUC = {best_auc_v:.3f}')
lines.append(f'  (Pilot reported H1=0.931; replication may differ.)')

summary = '\n'.join(lines)
with open(DETECT_SUMMARY, 'w') as f:
    f.write(summary)
print(summary)
print(f'\nSaved: {DETECT_SUMMARY}')
